# T13 — Explainable AI (SHAP) untuk Deteksi Pembukaan Lahan di Kawasan PSN Merauke

Pertanyaan: **fitur apa yang paling menentukan keputusan model**, apakah penjelasan itu stabil (antar-lipatan, antar-metode, antar-blok), dan apakah model ringkas dari fitur teratas tetap akurat?

| Tahap | Isi | Waktu |
|---|---|---|
| 0–2 | Instalasi, login, pemilihan **Blok G dan H** (peringkat berikutnya setelah Blok A–F) | 3 menit |
| 3 | 27 fitur dalam 6 kelompok: optik kondisi 2025, optik perubahan 2023→2025, radar kondisi, radar perubahan, topografi, tekstur | 1 menit |
| 4 | Label otomatis dari konsensus Hansen + RADD/TMF: dibuka, hutan stabil, lahan terbuka stabil (tanpa survei lapangan, tanpa penilaian manual) | 10–15 menit |
| 5 | XGBoost, Random Forest, regresi logistik: validasi silang spasial dan transfer antarblok | 3–5 menit |
| 6 | SHAP: kepentingan global, kelompok fitur, beeswarm, dependence; kesepakatan dengan 4 ukuran kepentingan lain; stabilitas | 5 menit |
| 7 | Model ringkas (top-k fitur SHAP) dan analisis kesalahan | 3 menit |
| 8 | Gambar dua bahasa, unduh `T13_hasil.zip` | 3 menit |

*Runtime → Jalankan semua*. Izinkan login Earth Engine dan Google Drive. Tidak ada penilaian manual. Hasil dan sampel disimpan di Google Drive `T13_xai/`; bila Colab terputus, jalankan lagi dan sampel tidak diunduh ulang.

In [ ]:
!pip install -q earthengine-api scikit-learn pandas matplotlib scipy openpyxl requests xgboost shap

In [ ]:
import ee
GEE_PROJECT = 'riset-psn-pps'
try:
    ee.Initialize(project=GEE_PROJECT)
except Exception:
    ee.Authenticate(force=True); ee.Initialize(project=GEE_PROJECT)
print('Earth Engine siap:', ee.String('OK').getInfo())

In [ ]:
import os, json, time, zipfile, requests, glob
import numpy as np, pandas as pd
TOPIC = 'T13'; SEED = 42
try:
    from google.colab import drive; drive.mount('/content/drive'); BASE = '/content/drive/MyDrive/T13_xai'
except Exception:
    BASE = '.'
os.makedirs(BASE, exist_ok=True)
OUT = os.path.join(BASE, f'{TOPIC}_hasil')                                   # disimpan di Google Drive agar aman bila Colab terputus
DIR_DATA, DIR_TAB, DIR_FIG = [os.path.join(OUT, d) for d in ('07_dataset', '03_tabel', '02_gambar')]
for d in (DIR_DATA, DIR_TAB, DIR_FIG): os.makedirs(d, exist_ok=True)
CRS = 'EPSG:32754'; BLOCK_KM = 20
PRE, POST = ('2023-06-01', '2023-11-01'), ('2025-06-01', '2025-11-01')       # musim kemarau
N_CLR, N_FOR, N_OPEN = 1500, 750, 750                                         # titik per kelas per blok
GROUP_M = 1000                                                                # blok spasial validasi silang (m)
USED = {'A': (140.546, -8.232), 'B': (140.7276, -8.2324), 'C': (140.7278, -7.8706), 'D': (139.2781, -7.5055),
        'E': (139.0969, -7.5047), 'F': (140.7285, -6.7851)}                   # blok topik lain
FEAT_GROUPS = {
    'Optik kondisi': ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI', 'NBR', 'NDMI', 'MNDWI', 'BSI'],
    'Optik perubahan': ['dNDVI', 'dNBR', 'dNDMI', 'dBSI', 'dMNDWI'],
    'Radar kondisi': ['VV', 'VH', 'VH_VV', 'VH_sd'],
    'Radar perubahan': ['dVV', 'dVH'],
    'Topografi': ['elev', 'slope'],
    'Tekstur': ['NDVI_sd', 'NIR_sd', 'dNDVI_m'],
}
FEAT = [f for g in FEAT_GROUPS.values() for f in g]; GROUP_OF = {f: g for g, fs in FEAT_GROUPS.items() for f in fs}
GROUP_EN = {'Optik kondisi': 'Optical state', 'Optik perubahan': 'Optical change', 'Radar kondisi': 'Radar state', 'Radar perubahan': 'Radar change',
            'Topografi': 'Terrain', 'Tekstur': 'Texture'}
MODELS = ['XGBoost', 'Random Forest', 'Regresi logistik']
COLS_EN = {'blok': 'block', 'model': 'model', 'skema': 'scheme', 'F1': 'F1', 'OA': 'OA', 'AUC': 'AUC', 'kappa': 'kappa', 'presisi': 'precision', 'recall': 'recall',
           'fitur': 'feature', 'kelompok': 'group', 'SHAP_rata2_abs': 'mean_abs_SHAP', 'peringkat': 'rank', 'ukuran': 'measure', 'rho_Spearman': 'Spearman_rho',
           'irisan_top5': 'top5_overlap', 'irisan_top10': 'top10_overlap', 'W_Kendall': 'Kendall_W', 'k_fitur': 'k_features', 'F1_CV': 'F1_CV', 'F1_transfer': 'F1_transfer',
           'jenis': 'type', 'jumlah': 'count', 'persen': 'percent', 'kelas_asal': 'source_class', 'lon': 'lon', 'lat': 'lat', 'loss_2024_2025_ha': 'loss_2024_2025_ha',
           'keterangan': 'note', 'arah': 'direction', 'persen_SHAP': 'SHAP_share_pct', 'nilai_p': 'p_value', 'pembanding': 'comparison', 'n': 'n',
           'rata2_SHAP_FP': 'mean_SHAP_FP', 'rata2_SHAP_TN': 'mean_SHAP_TN', 'rata2_SHAP_TP': 'mean_SHAP_TP', 'rata2_SHAP_FN': 'mean_SHAP_FN'}
VAL_EN = {**GROUP_EN, 'Regresi logistik': 'Logistic regression', 'Spasial 5 lipatan': 'Spatial 5-fold', 'Transfer': 'Transfer',
          'SHAP XGBoost': 'SHAP XGBoost', 'SHAP Random Forest': 'SHAP Random Forest', 'Gain XGBoost': 'XGBoost gain', 'Permutasi XGBoost': 'XGBoost permutation',
          '|Koefisien| logistik': '|Logistic coefficient|', 'Hutan stabil': 'Stable forest', 'Lahan terbuka stabil': 'Stable open land', 'Dibuka': 'Cleared',
          'Positif palsu': 'False positive', 'Negatif palsu': 'False negative', 'Antar-lipatan': 'Across folds', 'Antar-blok (G vs H)': 'Between blocks (G vs H)', 'Semua': 'All'}
def save_tab(df, name_id, name_en):
    d = df.copy()
    for c in d.columns:
        if not pd.api.types.is_numeric_dtype(d[c]): d[c] = d[c].map(lambda v: VAL_EN.get(v, v))
    df.to_csv(os.path.join(DIR_TAB, f'T13_tabel_{name_id}_ID.csv'), index=False)
    d.rename(columns=COLS_EN).to_csv(os.path.join(DIR_TAB, f'T13_table_{name_en}_EN.csv'), index=False)
np.random.seed(SEED); print('Folder keluaran:', OUT, '|', len(FEAT), 'fitur')

## Tahap 2 — Blok G dan H
Grid 20 × 20 km yang sama dengan topik lain; sel diurutkan menurut kehilangan tutupan pohon Hansen 2024–2025 (30 m). Sel Blok A–F dilewati; dua sel berikutnya menjadi **Blok G** (wilayah utama) dan **Blok H** (uji transfer dan stabilitas penjelasan).

In [ ]:
kab = ee.FeatureCollection('FAO/GAUL/2015/level2').filter(ee.Filter.eq('ADM2_NAME', 'Merauke')); kab_geom = kab.geometry()
gfc = ee.Image('UMD/hansen/global_forest_change_2025_v1_13'); lossyear = gfc.select('lossyear').unmask(0); tc2000 = gfc.select('treecover2000')
loss_ha = ee.Image.pixelArea().divide(1e4).updateMask(lossyear.gte(24).And(lossyear.lte(25))).rename('loss_ha')
grid = kab_geom.coveringGrid(ee.Projection(CRS).atScale(BLOCK_KM * 1000))
top = (loss_ha.reduceRegions(collection=grid, reducer=ee.Reducer.sum(), scale=30, tileScale=8).filter(ee.Filter.gt('sum', 0)).sort('sum', False)
       .limit(12).map(lambda f: f.set('cx', f.geometry().centroid(1).coordinates().get(0), 'cy', f.geometry().centroid(1).coordinates().get(1)))).getInfo()['features']
rows, BLK = [], {}
for i, f in enumerate(top):
    cx, cy = f['properties']['cx'], f['properties']['cy']
    used = [k for k, (ux, uy) in USED.items() if abs(cx - ux) < 0.05 and abs(cy - uy) < 0.05]
    rows.append({'peringkat': i + 1, 'lon': round(cx, 4), 'lat': round(cy, 4), 'loss_2024_2025_ha': round(f['properties']['sum'], 1), 'keterangan': f'dipakai Blok {used[0]}' if used else ''})
    if not used and len(BLK) < 2:
        b = 'GH'[len(BLK)]; BLK[b] = ee.Geometry(f['geometry']); rows[-1]['keterangan'] = f'Blok {b} (T13)'
blocks = pd.DataFrame(rows); print(blocks.to_string(index=False)); save_tab(blocks, 'kandidat_blok', 'candidate_blocks')
for b, g in BLK.items():
    json.dump({'type': 'FeatureCollection', 'features': [{'type': 'Feature', 'properties': {'blok': b}, 'geometry': g.transform('EPSG:4326', 1).getInfo()}]}, open(os.path.join(DIR_DATA, f'T13_blok{b}.geojson'), 'w'))
REG = BLK['G'].union(BLK['H'], 1).buffer(500)

## Tahap 3 — 27 fitur dalam 6 kelompok
| Kelompok | Fitur |
|---|---|
| Optik kondisi 2025 | B2, B3, B4, B8, B11, B12, NDVI, NBR, NDMI, MNDWI, BSI (median kemarau 2025, Sentinel-2 + Cloud Score+) |
| Optik perubahan | dNDVI, dNBR, dNDMI, dBSI, dMNDWI (2025 − 2023) |
| Radar kondisi | VV, VH, VH − VV (dB), simpangan baku VH 2025 |
| Radar perubahan | dVV, dVH (dB, 2025 − 2023) |
| Topografi | elevasi, kemiringan (Copernicus DEM GLO-30) |
| Tekstur | simpangan baku 3 × 3 NDVI dan NIR, rata-rata 3 × 3 dNDVI |

In [ ]:
cs_plus = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')
def s2(a, b):
    m = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(REG).filterDate(a, b).linkCollection(cs_plus, ['cs_cdf'])
         .map(lambda i: i.updateMask(i.select('cs_cdf').gte(0.6)).select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12']).divide(10000)).median())
    nd = lambda x, y, n: m.normalizedDifference([x, y]).rename(n)
    bsi = m.expression('((S + R) - (N + B)) / ((S + R) + (N + B))', {'S': m.select('B11'), 'R': m.select('B4'), 'N': m.select('B8'), 'B': m.select('B2')}).rename('BSI')
    return m.addBands([nd('B8', 'B4', 'NDVI'), nd('B8', 'B12', 'NBR'), nd('B8', 'B11', 'NDMI'), nd('B3', 'B11', 'MNDWI'), bsi])
def s1col(a, b):
    return (ee.ImageCollection('COPERNICUS/S1_GRD').filterBounds(REG).filterDate(a, b).filter(ee.Filter.eq('instrumentMode', 'IW'))
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')).select(['VV', 'VH']))
pre, post = s2(*PRE), s2(*POST)
d_opt = ee.Image.cat([post.select(k).subtract(pre.select(k)).rename('d' + k) for k in ('NDVI', 'NBR', 'NDMI', 'BSI', 'MNDWI')])
c1, c0 = s1col(*POST), s1col(*PRE); r1, r0 = c1.median(), c0.median()
radar = ee.Image.cat([r1.select('VV'), r1.select('VH'), r1.select('VH').subtract(r1.select('VV')).rename('VH_VV'), c1.select('VH').reduce(ee.Reducer.stdDev()).rename('VH_sd'),
                      r1.select('VV').subtract(r0.select('VV')).rename('dVV'), r1.select('VH').subtract(r0.select('VH')).rename('dVH')])
demc = ee.ImageCollection('COPERNICUS/DEM/GLO30').filterBounds(REG).select('DEM')
dem = demc.mosaic().setDefaultProjection(demc.first().projection()).rename('elev'); slope = ee.Terrain.slope(dem).rename('slope')
k3 = ee.Kernel.square(1)
tex = ee.Image.cat([post.select('NDVI').reduceNeighborhood(ee.Reducer.stdDev(), k3).rename('NDVI_sd'), post.select('B8').reduceNeighborhood(ee.Reducer.stdDev(), k3).rename('NIR_sd'),
                    d_opt.select('dNDVI').reduceNeighborhood(ee.Reducer.mean(), k3).rename('dNDVI_m')])
FEATIMG = ee.Image.cat([post.select(FEAT_GROUPS['Optik kondisi']), d_opt, radar, dem, slope, tex]).select(FEAT).toFloat()
print('Fitur:', FEATIMG.bandNames().getInfo())

## Tahap 4 — Label otomatis dan sampel
| Kelas | Aturan | Titik per blok |
|---|---|---|
| 1 Dibuka | Hansen kehilangan 2024/2025, dikonfirmasi RADD 2024–2025 atau JRC TMF Des-2025 terdeforestasi/terdegradasi | 1.500 |
| 2 Hutan stabil | tutupan pohon 2000 ≥ 50%, tanpa kehilangan s.d. 2025, tanpa RADD | 750 |
| 3 Lahan terbuka stabil | tutupan pohon 2000 < 10%, tanpa kehilangan, tanpa RADD | 750 |

Semua kelas: bukan air (JRC GSW < 50%). Model dilatih biner: **dibuka (1) vs tidak dibuka (2 + 3)**; kelas asal disimpan untuk analisis kesalahan.

In [ ]:
radd = (ee.ImageCollection('projects/radar-wur/raddalert/v1').filterMetadata('layer', 'contains', 'alert')
        .filterMetadata('geography', 'equals', 'asia').sort('system:time_end', False).first())
rdate = radd.select('Date').unmask(0); tmf25 = ee.ImageCollection('projects/JRC/TMF/v1_2025/AnnualChanges').mosaic().select('Dec2025')
gsw = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence').unmask(0)
clr = lossyear.gte(24).And(lossyear.lte(25)).And(rdate.gte(24001).And(rdate.lte(25366)).Or(tmf25.eq(2)).Or(tmf25.eq(3)))
noch = lossyear.eq(0).And(rdate.eq(0))
CLS = ee.Image(0).where(noch.And(tc2000.gte(50)), 2).where(noch.And(tc2000.lt(10)), 3).where(clr, 1).updateMask(gsw.lt(50)).rename('kelas').toInt()
CLS = CLS.updateMask(CLS.gt(0))
def fc_to_df(fc, page=1000):
    n = fc.size().getInfo(); lst = fc.toList(n); rows = []
    for s in range(0, n, page):
        for f in ee.FeatureCollection(lst.slice(s, s + page)).getInfo()['features']:
            p = dict(f['properties']); p['lon'], p['lat'] = f['geometry']['coordinates']; rows.append(p)
    return pd.DataFrame(rows)
F_S = os.path.join(DIR_DATA, 'T13_sampel.csv'); HANSEN = {}
for b, g in BLK.items(): HANSEN[b] = loss_ha.reduceRegion(ee.Reducer.sum(), g, 30, maxPixels=1e10, tileScale=8).get('loss_ha').getInfo()
if os.path.exists(F_S):
    S = pd.read_csv(F_S); print('Sampel dimuat ulang')
else:
    parts = []
    for b, g in BLK.items():
        t0 = time.perf_counter()
        fc = CLS.addBands(FEATIMG).stratifiedSample(numPoints=0, classBand='kelas', classValues=[1, 2, 3], classPoints=[N_CLR, N_FOR, N_OPEN], region=g, scale=10,
                                                     seed=SEED, geometries=True, tileScale=8, dropNulls=True)
        d = fc_to_df(fc); d['blok'] = b; parts.append(d); print(f'Blok {b}: {len(d)} titik ({time.perf_counter() - t0:.0f} dtk)')
    S = pd.concat(parts, ignore_index=True); S.to_csv(F_S, index=False)
S['y'] = (S.kelas == 1).astype(int)
lat0 = np.radians(S.lat.mean()); S['grp'] = (np.floor(S.lon * 111320 * np.cos(lat0) / GROUP_M).astype(int) * 100000 + np.floor(S.lat * 110540 / GROUP_M).astype(int))
print(S.groupby(['blok', 'kelas']).size().unstack())

## Tahap 5 — Tiga model: validasi silang spasial dan transfer
* **XGBoost** (model utama), **Random Forest**, dan **regresi logistik** (fitur distandardisasi) — hiperparameter tetap agar perbandingan penjelasan adil.
* Validasi silang 5 lipatan **spasial** (blok 1 km tidak terbelah) di setiap blok, dan **transfer**: latih di satu blok, uji di blok lain.

In [ ]:
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, cohen_kappa_score, precision_score, recall_score
def make(name):
    if name == 'XGBoost':
        return xgb.XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', n_jobs=-1, random_state=SEED)
    if name == 'Random Forest':
        return RandomForestClassifier(n_estimators=200, max_depth=12, min_samples_leaf=2, n_jobs=-1, random_state=SEED)
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=1.0))
def folds(d, k=5):
    ug = np.unique(d.grp); perm = np.random.default_rng(SEED).permutation(ug); f = {g: i % k for i, g in enumerate(perm)}; return d.grp.map(f).values
def mets(y, p, s): return dict(presisi=precision_score(y, p, zero_division=0), recall=recall_score(y, p), F1=f1_score(y, p), OA=accuracy_score(y, p), kappa=cohen_kappa_score(y, p), AUC=roc_auc_score(y, s))
D = {b: S[S.blok == b].reset_index(drop=True) for b in BLK}
for b in BLK: D[b]['fold'] = folds(D[b])
MET, OOF = [], {}
for b in BLK:
    d = D[b]; X, y = d[FEAT].values, d.y.values
    for mname in MODELS:
        s = np.zeros(len(d))
        for k in range(5):
            tr, te = d.fold.values != k, d.fold.values == k
            m = make(mname).fit(X[tr], y[tr]); s[te] = m.predict_proba(X[te])[:, 1]
        OOF[(b, mname)] = s; MET.append({'blok': b, 'model': mname, 'skema': 'Spasial 5 lipatan', **mets(y, (s >= 0.5).astype(int), s)})
for a, c in (('G', 'H'), ('H', 'G')):
    for mname in MODELS:
        m = make(mname).fit(D[a][FEAT].values, D[a].y.values); s = m.predict_proba(D[c][FEAT].values)[:, 1]
        MET.append({'blok': f'{a}→{c}', 'model': mname, 'skema': 'Transfer', **mets(D[c].y.values, (s >= 0.5).astype(int), s)})
MET = pd.DataFrame(MET).round(4); save_tab(MET, 'akurasi_model', 'model_accuracy'); print(MET.to_string(index=False))

## Tahap 6 — SHAP dan kesepakatan dengan ukuran kepentingan lain
* SHAP TreeExplainer untuk XGBoost (dan Random Forest), dilatih pada seluruh titik blok; nilai SHAP dihitung pada skala log-odds.
* Pembanding: **gain** XGBoost, **kepentingan permutasi** (penurunan F1 pada lipatan uji spasial), **|koefisien|** regresi logistik terstandardisasi, dan SHAP Random Forest.
* Stabilitas: W Kendall peringkat SHAP antar 5 model lipatan, dan korelasi Spearman peringkat SHAP Blok G vs H.

In [ ]:
import shap
def perm_imp(model, X, y, n_rep=10):
    # kepentingan permutasi: rata-rata penurunan F1 saat satu fitur diacak (ditulis sendiri agar tidak bergantung versi scikit-learn/xgboost)
    base = f1_score(y, model.predict_proba(X)[:, 1] >= 0.5); r = np.random.default_rng(SEED); out = np.zeros(X.shape[1])
    for j in range(X.shape[1]):
        dr = []
        for _ in range(n_rep):
            Xp = X.copy(); Xp[:, j] = r.permutation(Xp[:, j]); dr.append(base - f1_score(y, model.predict_proba(Xp)[:, 1] >= 0.5))
        out[j] = np.mean(dr)
    return out
from scipy.stats import spearmanr
def shap_vals(model, X):
    sv = shap.TreeExplainer(model).shap_values(X)
    if isinstance(sv, list): sv = sv[1]
    sv = np.asarray(sv)
    return sv[..., 1] if sv.ndim == 3 else sv
FULL, SV, IMP = {}, {}, {}
rng = np.random.default_rng(SEED)
for b in BLK:
    d = D[b]; X, y = d[FEAT].values, d.y.values
    mx = make('XGBoost').fit(X, y); mr = make('Random Forest').fit(X, y); ml = make('Regresi logistik').fit(X, y); FULL[b] = (mx, mr, ml)
    SV[b] = shap_vals(mx, X)
    sub = rng.choice(len(d), size=min(1000, len(d)), replace=False); svr = shap_vals(mr, X[sub])
    tr, te = d.fold.values != 0, d.fold.values == 0
    mpi = make('XGBoost').fit(X[tr], y[tr]); pi_imp = perm_imp(mpi, X[te], y[te])
    gain = mx.get_booster().get_score(importance_type='gain'); gain = np.array([gain.get(f'f{i}', 0.0) for i in range(len(FEAT))])
    IMP[b] = pd.DataFrame({'fitur': FEAT, 'kelompok': [GROUP_OF[f] for f in FEAT], 'SHAP XGBoost': np.abs(SV[b]).mean(0), 'SHAP Random Forest': np.abs(svr).mean(0),
                           'Gain XGBoost': gain, 'Permutasi XGBoost': pi_imp, '|Koefisien| logistik': np.abs(ml[-1].coef_[0])})
GI = []
for b in BLK:
    t = IMP[b].sort_values('SHAP XGBoost', ascending=False).reset_index(drop=True)
    for i, r in t.iterrows(): GI.append({'blok': b, 'peringkat': i + 1, 'fitur': r.fitur, 'kelompok': r.kelompok, 'SHAP_rata2_abs': r['SHAP XGBoost']})
GI = pd.DataFrame(GI).round(5); save_tab(GI, 'kepentingan_SHAP', 'SHAP_importance'); print(GI[GI.peringkat <= 10].to_string(index=False))
GRPT = []
for b in BLK:
    g = IMP[b].groupby('kelompok')['SHAP XGBoost'].sum(); tot = g.sum()
    for k, v in g.sort_values(ascending=False).items(): GRPT.append({'blok': b, 'kelompok': k, 'SHAP_rata2_abs': v, 'persen_SHAP': 100 * v / tot})
GRPT = pd.DataFrame(GRPT).round(3); save_tab(GRPT, 'kepentingan_kelompok', 'group_importance'); print(GRPT.to_string(index=False))
MEAS = ['SHAP XGBoost', 'SHAP Random Forest', 'Gain XGBoost', 'Permutasi XGBoost', '|Koefisien| logistik']
AGR = []
for b in BLK:
    t = IMP[b]; top = lambda c, k: set(t.nlargest(k, c).fitur)
    for c in MEAS[1:]:
        AGR.append({'blok': b, 'ukuran': c, 'rho_Spearman': spearmanr(t['SHAP XGBoost'], t[c])[0], 'irisan_top5': len(top('SHAP XGBoost', 5) & top(c, 5)), 'irisan_top10': len(top('SHAP XGBoost', 10) & top(c, 10))})
AGR = pd.DataFrame(AGR).round(3); save_tab(AGR, 'kesepakatan_ukuran', 'measure_agreement'); print(AGR.to_string(index=False))
def kendall_w(R):
    m, n = R.shape; Rj = R.sum(0); S_ = ((Rj - Rj.mean()) ** 2).sum(); return 12 * S_ / (m ** 2 * (n ** 3 - n))
STAB = []
for b in BLK:
    d = D[b]; X, y = d[FEAT].values, d.y.values; ranks = []
    for k in range(5):
        tr = d.fold.values != k; m = make('XGBoost').fit(X[tr], y[tr]); v = np.abs(shap_vals(m, X[tr])).mean(0); ranks.append(pd.Series(-v).rank().values)
    STAB.append({'blok': b, 'jenis': 'Antar-lipatan', 'W_Kendall': kendall_w(np.array(ranks)), 'rho_Spearman': np.nan, 'irisan_top5': np.nan, 'irisan_top10': np.nan})
tg, th = IMP['G'].set_index('fitur')['SHAP XGBoost'], IMP['H'].set_index('fitur')['SHAP XGBoost']
STAB.append({'blok': 'G vs H', 'jenis': 'Antar-blok (G vs H)', 'W_Kendall': np.nan, 'rho_Spearman': spearmanr(tg[FEAT], th[FEAT])[0],
             'irisan_top5': len(set(tg.nlargest(5).index) & set(th.nlargest(5).index)), 'irisan_top10': len(set(tg.nlargest(10).index) & set(th.nlargest(10).index))})
STAB = pd.DataFrame(STAB).round(3); save_tab(STAB, 'stabilitas_SHAP', 'SHAP_stability'); print(STAB.to_string(index=False))
pd.concat([IMP[b].assign(blok=b) for b in BLK]).round(6).to_csv(os.path.join(DIR_DATA, 'T13_semua_ukuran_kepentingan.csv'), index=False)
pd.concat([pd.DataFrame(SV[b], columns=FEAT).assign(blok=b, kelas=D[b].kelas.values, y=D[b].y.values) for b in BLK]).round(5).to_csv(os.path.join(DIR_DATA, 'T13_nilai_SHAP.csv'), index=False)

## Tahap 7 — Model ringkas dan analisis kesalahan
* **Model ringkas:** XGBoost dilatih ulang hanya dengan k fitur SHAP teratas (peringkat dari Blok G), k = 1, 2, 3, 5, 8, 12, 27; F1 validasi silang spasial di Blok G dan F1 transfer G → H.
* **Kesalahan:** dari prediksi lipatan uji XGBoost, positif palsu dibagi menurut kelas asal (hutan stabil / lahan terbuka stabil), lalu rata-rata SHAP fitur teratas dibandingkan antara positif palsu, positif benar, negatif benar, dan negatif palsu.

In [ ]:
rankG = GI[GI.blok == 'G'].sort_values('peringkat').fitur.tolist(); RED = []
for k in (1, 2, 3, 5, 8, 12, len(FEAT)):
    fs = rankG[:k]; d = D['G']; s = np.zeros(len(d))
    for f_ in range(5):
        tr, te = d.fold.values != f_, d.fold.values == f_
        s[te] = make('XGBoost').fit(d.loc[tr, fs].values, d.y.values[tr]).predict_proba(d.loc[te, fs].values)[:, 1]
    mt = make('XGBoost').fit(d[fs].values, d.y.values); st = mt.predict_proba(D['H'][fs].values)[:, 1]
    RED.append({'k_fitur': k, 'fitur': ', '.join(fs) if k <= 5 else f'{k} teratas', 'F1_CV': f1_score(d.y, s >= 0.5), 'AUC': roc_auc_score(d.y, s), 'F1_transfer': f1_score(D['H'].y, st >= 0.5)})
RED = pd.DataFrame(RED).round(4); save_tab(RED, 'model_ringkas', 'reduced_models'); print(RED.to_string(index=False))
ERR, ESH = [], []
for b in BLK:
    d = D[b]; p = (OOF[(b, 'XGBoost')] >= 0.5).astype(int); y = d.y.values
    typ = np.where((p == 1) & (y == 1), 'TP', np.where((p == 1) & (y == 0), 'FP', np.where((p == 0) & (y == 0), 'TN', 'FN')))
    for kel, nm in ((2, 'Hutan stabil'), (3, 'Lahan terbuka stabil')):
        m_ = d.kelas.values == kel; ERR.append({'blok': b, 'kelas_asal': nm, 'jenis': 'Positif palsu', 'n': int(m_.sum()), 'jumlah': int((typ[m_] == 'FP').sum()), 'persen': 100 * float((typ[m_] == 'FP').mean())})
    m_ = d.kelas.values == 1; ERR.append({'blok': b, 'kelas_asal': 'Dibuka', 'jenis': 'Negatif palsu', 'n': int(m_.sum()), 'jumlah': int((typ[m_] == 'FN').sum()), 'persen': 100 * float((typ[m_] == 'FN').mean())})
    top5 = GI[(GI.blok == b) & (GI.peringkat <= 5)].fitur.tolist(); sv = pd.DataFrame(SV[b], columns=FEAT)
    for f in top5:
        ESH.append({'blok': b, 'fitur': f, **{f'rata2_SHAP_{t}': float(sv.loc[typ == t, f].mean()) if (typ == t).any() else np.nan for t in ('TP', 'FP', 'TN', 'FN')}})
ERR = pd.DataFrame(ERR).round(2); save_tab(ERR, 'kesalahan', 'errors'); print(ERR.to_string(index=False))
ESH = pd.DataFrame(ESH).round(4); save_tab(ESH, 'SHAP_menurut_jenis_prediksi', 'SHAP_by_prediction_type'); print(ESH.to_string(index=False))

## Tahap 8 — Gambar dua bahasa dan unduhan

In [ ]:
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 9})
GCOL = dict(zip(FEAT_GROUPS, ['#4C78A8', '#F58518', '#54A24B', '#B279A2', '#9D755D', '#72B7B2']))
T = {'ID': dict(shap='Rata-rata |SHAP| (log-odds)', sv='Nilai SHAP (log-odds)', val='Nilai fitur (dinormalisasi)', low='rendah', high='tinggi', k='Jumlah fitur teratas (k)',
                f1cv='F1 validasi silang (G)', f1tr='F1 transfer (G → H)', grp=lambda g: g, blk='Blok', meas='Ukuran kepentingan', share='Porsi |SHAP| (%)'),
     'EN': dict(shap='Mean |SHAP| (log-odds)', sv='SHAP value (log-odds)', val='Feature value (normalised)', low='low', high='high', k='Number of top features (k)',
                f1cv='Cross-validation F1 (G)', f1tr='Transfer F1 (G → H)', grp=lambda g: GROUP_EN[g], blk='Block', meas='Importance measure', share='|SHAP| share (%)')}
for lang, t in T.items():
    fig, axs = plt.subplots(1, 2, figsize=(11, 5.5))
    for ax, b in zip(axs, BLK):
        g = GI[(GI.blok == b) & (GI.peringkat <= 15)].iloc[::-1]
        ax.barh(g.fitur, g.SHAP_rata2_abs, color=[GCOL[k] for k in g.kelompok]); ax.set_title(f'{t["blk"]} {b}'); ax.set_xlabel(t['shap'])
    from matplotlib.patches import Patch
    axs[1].legend(handles=[Patch(color=c, label=t['grp'](k)) for k, c in GCOL.items()], fontsize=7, loc='lower right')
    fig.tight_layout(); fig.savefig(os.path.join(DIR_FIG, f'T13_gambar_kepentingan_SHAP_{lang}.png'), dpi=300); plt.close(fig)
    fig, axs = plt.subplots(1, 2, figsize=(11, 5.5))
    for ax, b in zip(axs, BLK):
        top = GI[(GI.blok == b) & (GI.peringkat <= 12)].fitur.tolist()[::-1]; Xd = D[b][FEAT]
        for i, f in enumerate(top):
            v = SV[b][:, FEAT.index(f)]; x = Xd[f].values; q = np.nanpercentile(x, [2, 98]); c = np.clip((x - q[0]) / max(q[1] - q[0], 1e-9), 0, 1)
            jit = np.random.default_rng(i).normal(0, 0.12, len(v)); ax.scatter(v, i + jit, c=c, cmap='coolwarm', s=3, alpha=0.6, vmin=0, vmax=1, rasterized=True)
        ax.set_yticks(range(len(top))); ax.set_yticklabels(top); ax.axvline(0, color='k', lw=0.5); ax.set_xlabel(t['sv']); ax.set_title(f'{t["blk"]} {b}')
    sm = plt.cm.ScalarMappable(cmap='coolwarm'); cb = fig.colorbar(sm, ax=axs, fraction=0.02, pad=0.01, ticks=[0, 1]); cb.ax.set_yticklabels([t['low'], t['high']]); cb.set_label(t['val'])
    fig.savefig(os.path.join(DIR_FIG, f'T13_gambar_beeswarm_{lang}.png'), dpi=300, bbox_inches='tight'); plt.close(fig)
    top3 = GI[(GI.blok == 'G') & (GI.peringkat <= 3)].fitur.tolist(); fig, axs = plt.subplots(1, 3, figsize=(13, 4))
    for ax, f in zip(axs, top3):
        for b, mk in zip(BLK, ('o', '^')):
            ax.scatter(D[b][f], SV[b][:, FEAT.index(f)], s=4, alpha=0.4, marker=mk, label=f'{t["blk"]} {b}', rasterized=True)
        ax.axhline(0, color='k', lw=0.5); ax.set_xlabel(f); ax.set_ylabel(t['sv'])
    axs[0].legend(fontsize=8, markerscale=3); fig.tight_layout(); fig.savefig(os.path.join(DIR_FIG, f'T13_gambar_dependence_{lang}.png'), dpi=300); plt.close(fig)
    fig, axs = plt.subplots(1, 2, figsize=(11, 4))
    gp = GRPT.pivot(index='kelompok', columns='blok', values='persen_SHAP').reindex(list(FEAT_GROUPS))
    gp.index = [t['grp'](k) for k in gp.index]; gp.plot.barh(ax=axs[0], color=['#4C78A8', '#F58518']); axs[0].set_xlabel(t['share']); axs[0].invert_yaxis(); axs[0].legend(title=t['blk'], fontsize=8)
    axs[1].plot(RED.k_fitur, RED.F1_CV, 'o-', label=t['f1cv']); axs[1].plot(RED.k_fitur, RED.F1_transfer, 's--', label=t['f1tr']); axs[1].set_xscale('log')
    axs[1].set_xticks(RED.k_fitur); axs[1].set_xticklabels(RED.k_fitur); axs[1].set_xlabel(t['k']); axs[1].set_ylabel('F1'); axs[1].legend(fontsize=8); axs[1].grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(os.path.join(DIR_FIG, f'T13_gambar_kelompok_dan_model_ringkas_{lang}.png'), dpi=300); plt.close(fig)
print(sorted(os.listdir(DIR_FIG)))

In [ ]:
summary = {'blok': blocks.to_dict('records'), 'hansen_loss_ha': HANSEN, 'n_sampel': {f'{k[0]}_{k[1]}': int(v) for k, v in S.groupby(['blok', 'kelas']).size().items()},
           'akurasi': MET.to_dict('records'), 'kepentingan_top10': GI[GI.peringkat <= 10].to_dict('records'), 'kelompok': GRPT.to_dict('records'),
           'kesepakatan': AGR.to_dict('records'), 'stabilitas': STAB.to_dict('records'), 'model_ringkas': RED.to_dict('records'), 'kesalahan': ERR.to_dict('records')}
json.dump(summary, open(os.path.join(DIR_TAB, 'T13_hasil_ringkas.json'), 'w'), indent=2, ensure_ascii=False, default=lambda o: o.item() if hasattr(o, 'item') else str(o))
with pd.ExcelWriter(os.path.join(DIR_TAB, 'T13_semua_tabel.xlsx')) as xw:
    for f in sorted(glob.glob(os.path.join(DIR_TAB, '*.csv'))): pd.read_csv(f).to_excel(xw, sheet_name=os.path.basename(f)[4:-4][:31], index=False)
zpath = os.path.join(BASE, 'T13_hasil.zip')
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, fs_ in os.walk(OUT):
        for fn in fs_: z.write(os.path.join(root, fn), os.path.relpath(os.path.join(root, fn), os.path.dirname(OUT)))
print(MET.to_string(index=False))
try:
    from google.colab import files; files.download(zpath)
except Exception:
    print('Arsip:', zpath)

Selesai. Upload `T13_hasil.zip` ke Claude. File juga ada di Google Drive `T13_xai/T13_hasil.zip`.